# 4.2 Fachwerke: Steifigkeitsmatrix und Lösung

In Kapitel 4.1 haben wir das Fachwerk als Datensatz aufgebaut: Knotenkoordinaten,
Konnektivitätsmatrix und Kraftvektor. Jetzt berechnen wir die **globale
Steifigkeitsmatrix** und lösen das LGS

\begin{equation*}
\mathbf{K} \cdot \vec{u} = \vec{F}
\end{equation*}

nach dem Verschiebungsvektor $\vec{u}$ auf. Das Vorgehen ist dasselbe wie in
Kapitel 3: Matrix aufstellen, Lösbarkeit prüfen, `np.linalg.solve` aufrufen,
Probe durchführen. Neu ist nur, wie die Matrix aus den Stäben aufgebaut wird.

## Lernziele

* [ ] Sie können die globale Steifigkeitsmatrix $\mathbf{K}$ aus den
  Steifigkeitsblöcken der einzelnen Stäbe zusammensetzen.
* [ ] Sie können das LGS durch Elimination der Lagerknoten reduzieren
  und mit `np.linalg.solve` lösen.
* [ ] Sie können aus dem Verschiebungsvektor die Lagerkräfte zurückrechnen.

## Die Steifigkeitsmatrix aufbauen

Wir bauen das Fachwerk aus Kapitel 4.1 noch einmal als Datensatz auf.
Zusätzlich kommen jetzt die **Materialeigenschaften** ins Spiel: der
Elastizitätsmodul, der Stabdurchmesser und der daraus berechnete Querschnitt.
Aus Geometrie und Material berechnen wir anschließend die globale
Steifigkeitsmatrix $\mathbf{K}$.

### Wie bauen wir eine Steifigkeitsmatrix auf?

Der Aufbau einer Steifigkeitsmatrix erfolgt in 5 Schritten:

1. *Fachwerk-Setup*: Knotenkoordinaten, Konnektivität, Kräfte und
   Materialeigenschaften definieren.
2. *Geometrie eines einzelnen Stabs*: Länge und Winkel berechnen.
3. *Stabsteifigkeit*: Federkonstante $k = E A / L$ bestimmen.
4. *Steifigkeitsblock*: $\mathbf{b} = k\,\vec{e}\vec{e}^\top$ für einen
   einzelnen Stab aufstellen.
5. *Assemblierung*: Alle Steifigkeitsblöcke in die globale Matrix
   $\mathbf{K}$ eintragen.

**Schritt 1 - Fachwerk-Setup:**
Wir übernehmen die Geometrie, Lagerung und Kräfte aus Kapitel 4.1.
Neu hinzu kommen der **Elastizitätsmodul** $E$ und der **Querschnitt** $A$.
Alle Knotendaten verwenden die einheitliche Konvention: Knoten $n$ steht in
Zeile $n$ (Form `(anzahl_knoten, 2)`).

In [ ]:
import numpy as np

# --- Materialeigenschaften ---
elastizitaetsmodul = 2.1e11                          # Stahl in N/m²
durchmesser        = 1.0e-2                          # Stabdurchmesser in m
querschnitt        = np.pi * 0.25 * durchmesser**2   # Kreisquerschnitt in m²

# --- Fachwerk-Setup (wie in Kapitel 4.1) ---
# knoten_pos[n, :] = [x_n, y_n]: Knoten n steht in Zeile n
knoten_pos = np.array([
    [0.0,  0.0],   # Knoten 0: x = 0.0 m, y = 0.0 m
    [1.0,  1.0],   # Knoten 1: x = 1.0 m, y = 1.0 m
    [2.0,  0.0],   # Knoten 2: x = 2.0 m, y = 0.0 m
])
anzahl_knoten = knoten_pos.shape[0]   # Anzahl der Zeilen = Anzahl Knoten

lager_indizes = [0, 2]

verbindung = np.zeros((anzahl_knoten, anzahl_knoten))
verbindung[0, 1] = 1
verbindung[1, 2] = 1
verbindung = verbindung + verbindung.T

kraft_knoten = np.zeros((anzahl_knoten, 2))
kraft_knoten[1, 1] = -5000.                          # 5000 N nach unten an Knoten 1
kraft_vektor = kraft_knoten.flatten()

**Schritt 2 - Geometrie eines einzelnen Stabs:**
Bevor wir die Schleife über alle Stäbe schreiben, rechnen wir die nötige
Geometrie an einem konkreten Beispiel durch: Stab 0-1. Wir berechnen den
**Differenzvektor**, die **Länge** $L$ und den **Winkel** $\varphi$
gegenüber der $x$-Achse.

![Geometrie von Stab 0-1](pics/chap04_stabgeometrie.svg)

Geometrie von Stab 0-1: Der Differenzvektor zeigt von Knoten 0 nach Knoten 1
und hat die Komponenten $\Delta x$ und $\Delta y$. Daraus ergeben sich die
Stablänge $L = \|\Delta\mathbf{r}\|$ und der Winkel $\varphi$ gegenüber der
$x$-Achse. Stab 1-2 ist grau dargestellt, da er in diesem Schritt nicht im
Fokus steht. (Quelle: eigene Abbildung; Lizenz [CC BY-SA
4.0](https://creativecommons.org/licenses/by-sa/4.0))

In [ ]:
# Stab 0-1: Geometrie berechnen
i, j = 0, 1

# knoten_pos[j] - knoten_pos[i] liefert den Differenzvektor als 1D-Array [Δx, Δy]
differenz   = knoten_pos[j] - knoten_pos[i]           # Vektor von i nach j
staeblaenge = np.linalg.norm(differenz)               # Länge L in m
winkel      = np.arctan2(differenz[1], differenz[0])  # Winkel φ in rad

# Ausgabe
print(f"Stab {i}-{j}:")
print(f"  Differenzvektor : {differenz} m")
print(f"  Länge L         : {staeblaenge:.4f} m")
print(f"  Winkel φ        : {np.degrees(winkel):.1f}°")

Der Winkel $\varphi$ beschreibt, wie der Stab im globalen Koordinatensystem
liegt. Er bestimmt, wie die Stabkräfte später auf $x$- und $y$-Richtung
aufgeteilt werden.

**Schritt 3 - Stabsteifigkeit:**
Ein gerader Stab verhält sich entlang seiner Längsachse wie eine lineare Feder.
Die **Stabsteifigkeit** $k$ lässt sich direkt aus dem Hookeschen Gesetz
herleiten: Mit $\sigma = E\,\varepsilon$, $\sigma = F/A$ und
$\varepsilon = \Delta L / L$ folgt

\begin{equation*}
F = E\,A\,\frac{\Delta L}{L} = \underbrace{\frac{E\,A}{L}}_{k}\,\Delta L.
\end{equation*}

Je kürzer und dicker ein Stab, desto größer seine Steifigkeit $k$ — genau wie
bei einer Schraubenfeder.

![Analogie zwischen einem geraden Stab und einer linearen Feder](pics/chap04_federanalogie.svg)

Analogie zwischen einem geraden Stab und einer linearen Feder: Beide reagieren
auf eine Kraft $F$ mit einer Dehnung $\Delta L$, die proportional zu $F$ ist.
Die Federkonstante des Stabs ist $k = EA/L$: Je größer der Elastizitätsmodul
$E$ oder der Querschnitt $A$, desto steifer der Stab; je länger der Stab,
desto nachgiebiger. (Quelle: eigene Abbildung; Lizenz [CC BY-SA
4.0](https://creativecommons.org/licenses/by-sa/4.0))

In [ ]:
# Stabsteifigkeit k = E * A / L für Stab 0-1
staebsteifigkeit = elastizitaetsmodul * querschnitt / staeblaenge

# Ausgabe
print(f"Stabsteifigkeit k (Stab {i}-{j}): {staebsteifigkeit:.4e} N/m")

### Mini-Übung 1

Stab 0-1 hat die Länge $\sqrt{2}\,\text{m}$ und die berechnete Steifigkeit
`staebsteifigkeit`. Überlegen Sie zuerst im Kopf:

1. Um welchen Faktor ändert sich $k$, wenn der Durchmesser von $1\,\text{cm}$
   auf $2\,\text{cm}$ verdoppelt wird?
2. Überprüfen Sie Ihre Antwort mit Code.

In [ ]:
# Code-Zelle

**Schritt 4 - Steifigkeitsblock eines einzelnen Stabs:**
Der Stab ist nur entlang seiner eigenen Achse steif. Diese Richtung beschreiben
wir durch den **Einheitsvektor**

\begin{equation*}
\vec{e} =
\begin{pmatrix} \cos\varphi \\ \sin\varphi \end{pmatrix}.
\end{equation*}

![Verschiebung](pics/chap04_projektion.svg)

Die Verschiebung $\vec{u}_j$ an Knoten $j$ wird in zwei Anteile zerlegt:
Die Komponente $u^{\parallel}$ entlang der Stabachse $\vec{e}$ (orange) dehnt
oder staucht den Stab und erzeugt die Stabkraft
$\vec{F}_{j} = k\,u^{\parallel}\,\vec{e}$. Die senkrechte Komponente
$u^{\perp}$ (grau) dreht den Stab leicht, ändert seine Länge nicht und
leistet keinen Beitrag zur Stabkraft. Der gestrichelte Stab zeigt die
verformte Lage. (Quelle: eigene Abbildung; Lizenz [CC BY-SA
4.0](https://creativecommons.org/licenses/by-sa/4.0))

Den Steifigkeitsblock leiten wir in drei Schritten her:

1. **Projizieren**: Die Relativverschiebung $\vec{u}_j - \vec{u}_i$ der
   beiden Stabendknoten wird auf die Stabachse projiziert. Der parallele
   Anteil beschreibt die Längenänderung des Stabs:
   \begin{equation*}
   u^{\parallel} = \vec{e}^\top (\vec{u}_j - \vec{u}_i).
   \end{equation*}

2. **Kraft berechnen**: Aus der Längenänderung $u^{\parallel}$ folgt die
   Stabkraft in Richtung $\vec{e}$:
   \begin{equation*}
   \vec{F}_j = k\,u^{\parallel}\,\vec{e}
             = k\,\vec{e}\,\bigl(\vec{e}^\top (\vec{u}_j - \vec{u}_i)\bigr).
   \end{equation*}

3. **Zurückverteilen**: Wir schreiben das Ergebnis in Matrixform. Das
   Produkt $k\,\vec{e}\vec{e}^\top$ fassen wir als **Steifigkeitsblock**
   $\mathbf{b}$ zusammen:
   \begin{equation*}
   \mathbf{b}
   = k\,\vec{e}\vec{e}^\top
   = k
   \begin{pmatrix}
   \cos^2\varphi            & \cos\varphi\,\sin\varphi \\
   \cos\varphi\,\sin\varphi & \sin^2\varphi
   \end{pmatrix}.
   \end{equation*}

Damit schreibt sich die Kraft an Knoten $j$ als $\vec{F}_j = \mathbf{b}\,(\vec{u}_j - \vec{u}_i)$.
Nach dem dritten Newtonschen Gesetz (Aktion = Reaktion) wirkt an Knoten $i$
dieselbe Kraft mit umgekehrtem Vorzeichen: $\vec{F}_i = -\mathbf{b}\,(\vec{u}_j - \vec{u}_i)$.

Daraus folgt direkt, wie der Block $\mathbf{b}$ in die globale Matrix
eingetragen wird: **positives Vorzeichen** auf beiden Diagonalblöcken (der Stab
widersteht einer Verschiebung beider Endknoten gleichermaßen), **negatives
Vorzeichen** auf den Koppelblöcken (wenn sich $j$ von $i$ entfernt, zieht der
Stab $i$ in Richtung $j$, d.h. die zugehörige äußere Gleochgewichtskraft auf $i$
wirkt daher entgegen der Verschiebung von $j$, was dem negativen Vorzeichen des
Koppelblocks entspricht).

In [ ]:
# Einheitsvektor entlang der Stabachse
cos_w = np.cos(winkel)
sin_w = np.sin(winkel)

# Steifigkeitsblock b = k * e * e^T  (2x2-Matrix)
k_element = staebsteifigkeit * np.array([
    [cos_w**2,       sin_w * cos_w],
    [sin_w * cos_w,  sin_w**2     ],
])

# Ausgabe
print(f"Steifigkeitsblock k_element (Stab {i}-{j}) in kN/m:")
print(np.round(k_element * 1e-3, 3))

Die Matrix `k_element` ist symmetrisch. Der Diagonaleintrag $k\cos^2\varphi$
gibt den Steifigkeitsbeitrag in $x$-Richtung an, $k\sin^2\varphi$ den in
$y$-Richtung. Der Nebendiagonaleintrag $k\cos\varphi\sin\varphi$ beschreibt
die Kopplung zwischen beiden Richtungen.

**Schritt 5 - Assemblierung zur globalen Steifigkeitsmatrix:**
Wir wiederholen Schritte 2-4 in einer Schleife über alle Stabpaare und tragen
jeden Steifigkeitsblock an den vier zugehörigen Stellen in die globale Matrix
$\mathbf{K}$ ein: positiv auf den **Diagonalblöcken** beider Endknoten,
negativ auf den **Koppelblöcken**.

![Blockstruktur](pics/chap04_blockstruktur.svg)

Blockstruktur der globalen Steifigkeitsmatrix $\mathbf{K}$: Die
**Diagonalblöcke** $\mathbf{K}_{00}$, $\mathbf{K}_{11}$, $\mathbf{K}_{22}$
(dunkelblau) enthalten die aufsummierten Steifigkeitsblöcke aller an den
jeweiligen Knoten angrenzenden Stäbe. Die **Koppelblöcke**
$\mathbf{K}_{01}$, $\mathbf{K}_{10}$, $\mathbf{K}_{12}$, $\mathbf{K}_{21}$
(hellblau) beschreiben die Kopplung zwischen verbundenen Knoten (negatives
Vorzeichen). Die **Nullblöcke** $\mathbf{K}_{02}$ und $\mathbf{K}_{20}$
(grau) zeigen, dass zwischen Knoten 0 und Knoten 2 kein Stab existiert. Jeder
Block hat die Dimension $2 \times 2$, weil jeder Knoten zwei Freiheitsgrade
($x$ und $y$) besitzt. (Quelle: eigene Abbildung; Lizenz [CC BY-SA
4.0](https://creativecommons.org/licenses/by-sa/4.0))

In [ ]:
# --- Globale Steifigkeitsmatrix: (2*anzahl_knoten) x (2*anzahl_knoten) ---
steifigkeit_global = np.zeros((2 * anzahl_knoten, 2 * anzahl_knoten))

for i in range(anzahl_knoten):
    for j in range(i + 1, anzahl_knoten):
        if verbindung[i, j]:
            # Schritt 2: Geometrie
            differenz        = knoten_pos[j] - knoten_pos[i]
            staeblaenge      = np.linalg.norm(differenz)
            winkel           = np.arctan2(differenz[1], differenz[0])

            # Schritt 3: Stabsteifigkeit
            staebsteifigkeit = elastizitaetsmodul * querschnitt / staeblaenge

            # Schritt 4: Steifigkeitsblock b = k * e * e^T
            cos_w     = np.cos(winkel)
            sin_w     = np.sin(winkel)
            k_element = staebsteifigkeit * np.array([
                [cos_w**2,       sin_w * cos_w],
                [sin_w * cos_w,  sin_w**2     ],
            ])

            # Schritt 5: In globale Matrix eintragen
            # Diagonalblöcke: +b (Stab widersteht Verschiebung beider Endknoten)
            steifigkeit_global[2*i : 2*(i+1), 2*i : 2*(i+1)] += k_element
            steifigkeit_global[2*j : 2*(j+1), 2*j : 2*(j+1)] += k_element
            # Koppelblöcke: -b (Aktion = Reaktion, entgegengesetztes Vorzeichen)
            steifigkeit_global[2*i : 2*(i+1), 2*j : 2*(j+1)] -= k_element
            steifigkeit_global[2*j : 2*(j+1), 2*i : 2*(i+1)] -= k_element

# Ausgabe
print("Globale Steifigkeitsmatrix K (in kN/m):")
print(np.round(steifigkeit_global * 1e-3, 3))

### Mini-Übung 2

1. `steifigkeit_global` hat die Dimension $6 \times 6$. Warum genau 6
   und nicht 3? Begründen Sie ohne Code.
2. Rufen Sie `np.linalg.det(steifigkeit_global)` auf. Was ergibt sich,
   und warum ist das erwartet? Überlegen Sie zuerst, was physikalisch
   passiert, wenn keine Lager gesetzt sind.
3. Der Block `steifigkeit_global[2:4, 2:4]` enthält die Freiheitsgrade
   von Knoten 1. Geben Sie diesen Block aus. Sind die beiden
   Diagonaleinträge gleich oder verschieden, und warum ist das für dieses
   Fachwerk plausibel?

In [ ]:
# Code-Zelle

## Lager einbauen, lösen und Lagerkräfte berechnen

Die globale Steifigkeitsmatrix $\mathbf{K}$ ist ohne Lager singulär: das
Fachwerk könnte als Starrkörper beliebig verschoben werden. Wir müssen daher
die Lagerknoten aus dem LGS entfernen, bevor wir es lösen können. Das
geschieht in fünf Schritten.

### Wie lösen wir das LGS?

1. *Freie Freiheitsgrade bestimmen*: Lagerknoten ausschließen, DOF-Indizes
   der freien Knoten sammeln.

2. *LGS reduzieren*: Steifigkeitsmatrix und Kraftvektor auf die freien
   Freiheitsgrade einschränken.

3. *Lösen*: `np.linalg.solve` auf das reduzierte System anwenden.
4. *Verschiebungsvektor rekonstruieren*: Lagerknoten auf null setzen,
   freie DOFs eintragen.

5. *Lagerkräfte zurückrechnen*: $\vec{F} = \mathbf{K} \cdot \vec{u}$
   für alle Knoten auswerten.

**Schritt 1 - Freie Freiheitsgrade bestimmen:**
Jeder Knoten hat zwei **Freiheitsgrade** (DOFs): einen in $x$- und einen in
$y$-Richtung. In diesem vereinfachten Modell werden bei jedem Lagerknoten
**beide** DOFs gesperrt (Festlager-Annahme: $u_x = u_y = 0$). Die
verbleibenden Knoten heißen **freie Knoten**; ihre DOFs bilden das eigentliche
Gleichungssystem.

Hinweis: In der Praxis wäre Knoten 2 oft als Loslager ausgeführt, das nur $u_y$
sperrt (z. B. ein Rollenlager). Dann wäre der $x$-DOF von Knoten 2 frei, und das
reduzierte LGS hätte eine Zeile mehr. Für dieses Einführungsbeispiel nehmen wir
vereinfachend an, dass beide Lager alle Freiheitsgrade sperren.

Wir sammeln zunächst die Indizes der freien Knoten und daraus die
zugehörigen DOF-Indizes in `freie_dofs`. Knoten $n$ trägt die
DOF-Indizes $2n$ (x-Richtung) und $2n+1$ (y-Richtung).

In [ ]:
# Freie Knoten: alle Knoten, die nicht gelagert sind
freie_indizes = []
for n in range(anzahl_knoten):
    if n not in lager_indizes:
        freie_indizes.append(n)

# Freie DOFs: jeder freie Knoten trägt DOF 2n (x) und 2n+1 (y)
freie_dofs = []
for n in freie_indizes:
    freie_dofs.append(2 * n)       # x-Freiheitsgrad
    freie_dofs.append(2 * n + 1)   # y-Freiheitsgrad
freie_dofs = np.array(freie_dofs)

# Ausgabe
print("Alle Knoten:  ", list(range(anzahl_knoten)))
print("Lagerknoten:  ", lager_indizes)
print("Freie Knoten: ", freie_indizes)
print("Freie DOFs:   ", freie_dofs)

`freie_dofs` enthält hier nur die Indizes `[2, 3]`, weil Knoten 1
der einzige freie Knoten ist. Das reduzierte LGS hat damit nur
$2 \times 2$ Einträge, also genau zwei Unbekannte für die $x$- und
$y$-Verschiebung von Knoten 1.

**Schritt 2 - LGS reduzieren:**
Wir schränken die globale Steifigkeitsmatrix $\mathbf{K}$ und den Kraftvektor
$\vec{F}$ auf die freien DOFs ein. Die Zeilen und Spalten der Lagerknoten
werden dabei weggelassen, weil ihre Verschiebungen bereits bekannt sind
(nämlich null).

In [ ]:
# Reduzierter Kraftvektor: nur Einträge der freien DOFs
kraft_reduziert = kraft_vektor[freie_dofs]

# Reduzierte Steifigkeitsmatrix: nur Zeilen und Spalten der freien DOFs
steifigkeit_reduziert = steifigkeit_global[freie_dofs, :][:, freie_dofs]

# Ausgabe
print(f"Reduzierte Steifigkeitsmatrix "
      f"({steifigkeit_reduziert.shape[0]}x{steifigkeit_reduziert.shape[1]})"
      f" in kN/m:")
print(np.round(steifigkeit_reduziert * 1e-3, 3))
print(f"Reduzierter Kraftvektor: {kraft_reduziert} N")

Das reduzierte LGS hat hier die Dimension $2 \times 2$, weil nur Knoten 1
frei ist. In größeren Fachwerken mit mehr freien Knoten wächst die
reduzierte Matrix entsprechend. Der Code ändert sich dabei nicht, weil
`freie_dofs` die Reduktion vollständig steuert.

**Schritt 3 - Lösen:**
Das reduzierte LGS

$$
\mathbf{K}_\text{red} \cdot \vec{u}_\text{red} = \vec{F}_\text{red}
$$

lösen wir mit `np.linalg.solve`, genau wie in Kapitel 3. Anschließend
führen wir eine Probe durch, um sicherzustellen, dass das Ergebnis korrekt
ist.

In [ ]:
# Lösen: K_red * u_red = F_red
verschiebung_reduziert = np.linalg.solve(steifigkeit_reduziert,
                                          kraft_reduziert)

# Probe: K_red * u_red muss gleich F_red ergeben
probe = np.allclose(steifigkeit_reduziert @ verschiebung_reduziert,
                    kraft_reduziert)

# Ausgabe
print("Verschiebungen der freien Knoten:")
for k, n in enumerate(freie_indizes):
    ux = verschiebung_reduziert[2 * k]
    uy = verschiebung_reduziert[2 * k + 1]
    print(f"  Knoten {n}: ux = {ux*1e3:.4f} mm,  uy = {uy*1e3:.4f} mm")

print(f"Probe bestanden: {probe}")

Das negative Vorzeichen von $u_y$ an Knoten 1 ist physikalisch sinnvoll:
Die Last zeigt nach unten, also bewegt sich der freie Knoten nach unten.
Die Verschiebung in $x$-Richtung ist null, weil die Geometrie des Fachwerks
symmetrisch ist und sich die Horizontalkomponenten beider Stäbe genau
aufheben.

**Schritt 4 - Vollständigen Verschiebungsvektor rekonstruieren:**
Die Lösung `verschiebung_reduziert` enthält nur die Verschiebungen der
freien Knoten. Wir bauen daraus den vollständigen Verschiebungsvektor
`verschiebung_gesamt` auf, der alle Knoten enthält. Die Lagerknoten
erhalten die Verschiebung null (Randbedingung).

In [ ]:
# Vollständiger Verschiebungsvektor: alle DOFs, Lager = 0
verschiebung_gesamt = np.zeros(2 * anzahl_knoten)

# Freie DOFs aus der Lösung eintragen
verschiebung_gesamt[freie_dofs] = verschiebung_reduziert

# Ausgabe
print("Vollständiger Verschiebungsvektor:")
for n in range(anzahl_knoten):
    ux = verschiebung_gesamt[2 * n]
    uy = verschiebung_gesamt[2 * n + 1]
    gelagert = n in lager_indizes
    print(f"  Knoten {n} (gelagert: {gelagert}): "
          f"ux = {ux*1e3:.4f} mm,  uy = {uy*1e3:.4f} mm")

Der Verschiebungsvektor hat dieselbe Struktur wie der Kraftvektor:
$2 \cdot n_\text{Knoten}$ Einträge, paarweise nach Knoten geordnet.
Er ist die Grundlage für die Lagerkraftberechnung in Schritt 5 und
für die Visualisierung der verformten Lage.

**Schritt 5 - Lagerkräfte zurückrechnen:**
Mit dem vollständigen Verschiebungsvektor $\vec{u}$ können wir alle
Knotenkräfte zurückrechnen, auch an den Lagerknoten. Wir nutzen dazu die
globale Gleichgewichtsbedingung

\begin{equation*}
\vec{F} = \mathbf{K} \cdot \vec{u}.
\end{equation*}

An den freien Knoten liefert das die aufgebrachten äußeren Kräfte zurück
(Probe). An den Lagerknoten liefert es die **Lagerreaktionen**, also die
Kräfte, die das Lager aufbringen muss, um das Fachwerk im Gleichgewicht
zu halten.

In [ ]:
# Alle Knotenkräfte zurückrechnen (inkl. Lagerreaktionen)
kraft_ergebnis = np.round(steifigkeit_global @ verschiebung_gesamt, 6)

# Ausgabe
print("Knotenkräfte (äußere Lasten und Lagerreaktionen):")
for n in range(anzahl_knoten):
    fx = kraft_ergebnis[2 * n]
    fy = kraft_ergebnis[2 * n + 1]
    gelagert = n in lager_indizes
    print(f"  Knoten {n} (gelagert: {gelagert}): "
          f"Fx = {fx:.1f} N,  Fy = {fy:.1f} N")

# Gleichgewichtsprüfung: Summe aller Knotenkräfte muss null ergeben
summe_fx = np.sum(kraft_ergebnis[0::2])
summe_fy = np.sum(kraft_ergebnis[1::2])
print(f"\nSumme aller Kräfte: Fx = {summe_fx:.1f} N,  Fy = {summe_fy:.1f} N")

Die Summe aller Knotenkräfte muss null ergeben, weil das Fachwerk im
Gleichgewicht ist. Die Lagerreaktionen an Knoten 0 und 2 heben zusammen die
äußere Last von $5000\,\text{N}$ nach unten auf. Wegen der Symmetrie des
Fachwerks trägt jedes Lager genau die Hälfte, also $2500\,\text{N}$ nach oben.
Zusätzlich treten horizontale Lagerreaktionen von $\pm 2500 N$ auf: Stab 0-1
drückt Knoten 0 nach links, Stab 1-2 drückt Knoten 2 nach rechts. In der Summe
heben sie sich auf.

### Mini-Übung 3

1. Addieren Sie die $y$-Lagerkräfte an Knoten 0 und Knoten 2. Was ergibt
   die Summe, und warum ist das die erwartete Antwort?
2. Verdoppeln Sie die Last auf $-10\,000$ N und berechnen Sie die
   Verschiebungen neu. Um welchen Faktor ändert sich `verschiebung_gesamt`?
   Begründen Sie den Zusammenhang ohne Code (Stichwort: Linearität des LGS).
3. Erhöhen Sie den Stabdurchmesser auf `durchmesser = 2e-2` m (alles andere
   bleibt gleich). Wie ändert sich die Absenkung von Knoten 1, und warum
   genau um diesen Faktor?

In [ ]:
# Hier Ihren Code eingeben

## Zusammenfassung und Ausblick

Wir haben die globale Steifigkeitsmatrix $\mathbf{K}$ aus den
Steifigkeitsblöcken der einzelnen Stäbe assembliert, das LGS durch
Elimination der Lagerknoten reduziert und mit `np.linalg.solve` gelöst.
Die Lagerreaktionen folgen direkt aus $\vec{F} = \mathbf{K} \cdot \vec{u}$.

Im nächsten Kapitel berechnen wir die Stabkräfte und visualisieren die
verformte Lage.